# Unit 6: 迁移学习与微调

## 学习目标
- 理解迁移学习的核心思想
- 掌握两种迁移学习策略：特征提取 vs 微调
- 学会使用 torchvision 预训练模型
- 掌握冻结/解冻层的技巧
- 实战：用 ResNet 预训练模型解决自定义图像分类

## 6.1 为什么需要迁移学习？

从零训练一个深度 CNN 需要：
- **海量数据** (ImageNet 有 120 万张)
- **大量算力** (在 ImageNet 上训练 ResNet-50 需要数天)
- **调参经验**

迁移学习的思想：
> 将一个在大规模数据集上训练好的模型，迁移到你的目标任务上。

**类比**：学会骑自行车后，学摩托车会更快——底层技能（平衡感）是可迁移的。

CNN 的层级特征：
- **浅层**：边缘、颜色、纹理（通用特征，可迁移）
- **中层**：形状、图案（较通用）
- **深层**：语义概念、物体部件（任务相关）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import copy

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## 6.2 两种迁移学习策略

### 策略 A：特征提取 (Feature Extraction)
- **冻结**预训练模型的所有参数
- 只训练新加的分类头
- 适用场景：数据量小，与预训练任务相似

### 策略 B：微调 (Fine-tuning)
- **解冻**预训练模型的部分或全部参数
- 以很小的学习率继续训练
- 适用场景：数据量中等，或与预训练任务有差异

### 策略选择建议

| 数据量 \ 相似度 | 高相似度 | 低相似度 |
|:---:|:---:|:---:|
| **小** | 特征提取 | 特征提取 (可能需要微调浅层) |
| **大** | 微调少量层 | 微调更多层或从头训练 |

## 6.3 torchvision 预训练模型概览

PyTorch 提供了丰富的预训练模型（均在 ImageNet 上训练）：

| 模型 | Top-1 Acc | 参数量 | 特点 |
|------|-----------|--------|------|
| ResNet-18 | 69.8% | 11.7M | 轻量，适合入门 |
| ResNet-50 | 76.1% | 25.6M | 经典，广泛使用 |
| ResNet-152 | 78.3% | 60.2M | 准确率最高 |
| EfficientNet-B0 | 77.1% | 5.3M | 效率极高 |
| MobileNetV3-Large | 74.0% | 5.5M | 移动端优化 |
| ConvNeXt-Tiny | 82.5% | 28.6M | SOTA |

> 注意：不同版本的 torchvision 模型名称可能不同，建议查看[官方文档](https://pytorch.org/vision/stable/models.html)。

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

available_weights = {
    "resnet18": models.ResNet18_Weights,
    "resnet50": models.ResNet50_Weights,
    "mobilenet_v3_large": models.MobileNet_V3_Large_Weights,
    "efficientnet_b0": models.EfficientNet_B0_Weights,
}

for name, weights_cls in available_weights.items():
    try:
        weights = weights_cls.DEFAULT
        print(f"{name:25s}: available")
    except Exception as e:
        print(f"{name:25s}: NOT available ({e})")

## 6.4 实战：特征提取模式

用预训练 ResNet-18 在 CIFAR-10 上做特征提取。  
1.获取默认权重枚举  
2.提取预处理 Transform  

ImageClassification(...) 是 Compose(...) 的新一代封装，功能等价但更灵活，无需担心兼容性问题。  
```python
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

# ========== 环节1: 模型加载（只管结构和权重）==========
weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)

# ========== 环节2: 获取配套 transform ==========
pretrained_transform = weights.transforms()

# ========== 环节3: 👉 在 Dataset 中配置 transform ==========
train_dataset = CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=pretrained_transform      # ← 在这里传入！
)

val_dataset = CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=pretrained_transform      # ← 验证集也用同一个
)

# ========== 环节4: DataLoader 只负责批处理 ==========
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
```

In [ ]:
# 加载ResNet18预训练模型的默认权重配置
weights = models.ResNet18_Weights.DEFAULT
# 获取与该预训练权重配套的标准图像预处理变换（包括Resize, CenterCrop, Normalize等）
pretrained_transform = weights.transforms()

print(f"预训练使用的预处理:\n{pretrained_transform}")

“加载带预训练权重的模型”（或“实例化预训练模型”）。

In [ ]:
# 加载预训练的ResNet18模型，weights参数指定使用预训练权重
model = models.resnet18(weights=weights)

print(f"原始最后一层: {model.fc}")

# 获取原始全连接层的输入特征数
num_features = model.fc.in_features
# 将原始全连接层替换为新的全连接层，输出类别数为10（适用于CIFAR-10等10分类任务）
model.fc = nn.Linear(num_features, 10)

print(f"新最后一层: {model.fc}")

# 冻结所有层的参数，使其在训练时不更新
for name, param in model.named_parameters():
    param.requires_grad = False

# 解冻最后一层全连接层的参数，使其在训练时更新
for name, param in model.fc.named_parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
# : 后跟 , 表示千分位分隔符
print(f"可训练参数: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

⚠️ 微调常识：当目标数据集（CIFAR-10）与预训练数据集（ImageNet）差异较大时，使用目标数据集自己的 mean/std 进行归一化，通常能让模型更快收敛、获得更好的精度。因为 BatchNorm 层在微调时会重新适应新的数据分布，输入数据的归一化基准也应与之匹配。


In [ ]:
model = model.to(device)

cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

train_size = 45000
val_size = 5000
train_set, val_set = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train: {train_size:,}, Val: {val_size:,}, Test: {len(test_dataset):,}")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for data, target in tqdm(loader, desc="Train", leave=False):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += data.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    for data, target in tqdm(loader, desc="Eval", leave=False):
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += data.size(0)
    return total_loss / total, correct / total

In [ ]:
criterion = nn.CrossEntropyLoss()
# 使用 Adam 优化器，只优化全连接层 (fc) 的参数，学习率设为 0.001
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

print("特征提取模式训练 (只训练 FC 层)...")
for epoch in range(5):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/5 | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

## 6.5 实战：微调模式

现在解冻部分层，以更小的学习率微调。  
ft 是 Fine-Tuning（微调） 的缩写。

| 层 | 提取的特征类型 | 跨数据集通用性 | 是否需要微调 |
| :--- | :--- | :--- | :--- |
| `conv1`, `layer1` | 边缘、纹理、颜色等底层视觉基元 | 🟢 极高（几乎与任务无关） | ❌ 冻结 |
| `layer2`, `layer3` | 局部部件（眼睛、轮子、条纹等） | 🟡 中等 | ⚠️ 视数据量而定 |
| `layer4` | 高层语义概念（物体整体、类别判别特征） | 🔴 低（高度耦合 ImageNet 的 1000 类） | ✅ 通常需要微调 |
| `fc` | 1000 维分类头 | 🔴 完全不适用 | ✅ 必须替换+训练 |



| 模块 | 输出通道数 | 参数量 | 占总量比例 | 累计 |
| :--- | :--- | :--- | :--- | :--- |
| `conv1` + `bn1` | 64 | ~9,408 | 0.1% | 0.1% |
| `layer1` (×2) | 64 | ~147,968 | 1.3% | 1.4% |
| `layer2` (×2) | 128 | ~525,568 | 4.7% | 6.1% |
| `layer3` (×2) | 256 | ~2,101,248 | 18.8% | 24.9% |
|* `layer4` (×2) | 512 | ~8,389,376 | 75.0% | 99.9% |
|* `fc` | 512→10 | ~5,130 | 0.05% | 100% |
| 总计 | | ~11,181,642 | | |



In [ ]:
model_ft = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_features, 10)
model_ft = model_ft.to(device)

print("layer4 及 fc 可训练，其余冻结")
for name, param in model_ft.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
        print(f"  Trainable: {name}")
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_ft.parameters())
print(f"\n可训练: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

In [ ]:
params_to_update = [p for p in model_ft.parameters() if p.requires_grad]
# 使用Adam优化器，学习率设为0.0001，仅更新需要梯度的参数
optimizer_ft = optim.Adam(params_to_update, lr=0.0001)
criterion = nn.CrossEntropyLoss()

print("微调模式训练 (layer4 + fc)...")
for epoch in range(5):
    train_loss, train_acc = train_one_epoch(model_ft, train_loader, criterion, optimizer_ft, device)
    val_loss, val_acc = evaluate(model_ft, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/5 | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

## 6.6 测试集最终评估

In [ ]:
test_loss, test_acc = evaluate(model_ft, test_loader, criterion, device)
print(f"\n{'='*50}")
print(f"微调 ResNet-18 在 CIFAR-10 上的测试结果:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc:  {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"{'='*50}")

print(f"\n对比:")
print(f"  Unit 4 自训练 CNN:     ~75-80%")
print(f"  Unit 5 MiniResNet:     ~85-88%")
print(f"  迁移学习 ResNet-18:    ~90-93%")

## 6.7 自定义分类头的多种设计

除了简单的 `nn.Linear`，你还可以设计更复杂的分类头。在骨干网络提取的特征和最终分类结果之间，增加一个可学习的"特征适配与正则化"模块，从而提升微调效果。

| 设计要素 | 单层 Linear | ClassifierHead | 作用 |
| :--- | :--- | :--- | :--- |
| 模型容量 | 仅 1 个线性变换 | 3 层 MLP + BN + ReLU | 能学习更复杂的决策边界，适配新任务 |
| 正则化 | ❌ 无 | ✅ Dropout × 2 + BN × 2 | 大幅缓解小数据集微调时的过拟合 |
| 特征维度 | 直接 512→10 | 512→512→256→10 | 瓶颈层（256）迫使模型学习紧凑、鲁棒的类别表征 |
| 梯度流 | 直接反传 | BN 稳定梯度 + Dropout 噪声注入 | 训练更稳定，尤其当 backbone 学习率较小时 |



In [ ]:
class ClassifierHead(nn.Module):
    def __init__(self, in_features, num_classes, hidden_dim=512, dropout=0.5):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, x):
        return self.fc(x)

model_custom_head = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = model_custom_head.fc.in_features
model_custom_head.fc = ClassifierHead(num_features, 10)
model_custom_head = model_custom_head.to(device)
print(f"自定义分类头:\n{model_custom_head.fc}")

In [ ]:
print("layer4 及 fc 可训练，其余冻结")
for name, param in model_custom_head.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
        print(f"  Trainable: {name}")
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model_custom_head.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_custom_head.parameters())
print(f"\n可训练: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

In [ ]:
params_to_update = [p for p in model_custom_head.parameters() if p.requires_grad]
# ✅ 正确做法：分层学习率
optimizer = optim.Adam([
    {"params": model_custom_head.layer4.parameters(), "lr": 1e-5},  # 微调
    {"params": model_custom_head.fc.parameters(),     "lr": 1e-3},  # 从头训练
])
criterion = nn.CrossEntropyLoss()

print("微调模式训练 (layer4 + fc)...")
for epoch in range(5):
    train_loss, train_acc = train_one_epoch(model_custom_head, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model_custom_head, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/5 | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
test_loss, test_acc = evaluate(model_custom_head, test_loader, criterion, device)
print(f"\n{'='*50}")
print(f"微调 ResNet-18 在 CIFAR-10 上的测试结果:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc:  {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"{'='*50}")

## 6.8 冻结与解冻的工具函数

In [ ]:
def set_requires_grad(model, requires_grad):
    for param in model.parameters():
        param.requires_grad = requires_grad

def freeze_all(model):
    set_requires_grad(model, False)

def unfreeze_all(model):
    set_requires_grad(model, True)

def freeze_until(model, layer_name):
    freeze = True
    for name, param in model.named_parameters():
        if layer_name in name:
            freeze = False
        param.requires_grad = not freeze

print("工具函数定义:")
print("  freeze_all(model)       - 冻结全部参数")
print("  unfreeze_all(model)     - 解冻全部参数")
print("  freeze_until(model, 'layer4') - 冻结 layer4 之前的所有层")

## 6.9 最佳实践总结

1. **始终使用预训练权重**，除非你的数据极其特殊
2. **先特征提取，再微调**：先用较大学习率训练分类头，再解冻微调
3. **微调时使用小学习率**：通常是特征提取的 1/10
4. **逐层解冻**：从深层到浅层逐步解冻，效果更好
5. **使用预训练模型的 transforms**：`weights.transforms()` 获取正确的预处理
6. **注意输入尺寸**：ImageNet 预训练模型通常需要 224x224
7. **BatchNorm 在微调时**：小 batch 时考虑冻结 BN 的 running stats

## 6.10 单元小结

| 概念 | 要点 |
|------|------|
| **迁移学习** | 利用大规模预训练模型的知识 |
| **特征提取** | 冻结骨干网络，只训练分类头 |
| **微调** | 解冻部分层，用小学习率继续训练 |
| **预训练模型** | `models.resnet18(weights=DEFAULT)` |
| **参数冻结** | `param.requires_grad = False` |

### 思考题
1. 为什么微调时要用比特征提取更小的学习率？
2. 如果目标任务和 ImageNet 差异很大（如医学影像），应该怎么做？
3. 什么时候应该从头训练而不是迁移学习？